In [ ]:
!nvidia-smi    

In [ ]:
!pip install -q uv

In [ ]:
!uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" bm25s rank-bm25 pyvi jieba loguru typer pydantic pyyaml tqdm flagembedding sentence-transformers trafilatura httpx qdrant_client

In [ ]:
import os                                                                                                                                                                             
from pathlib import Path                                                                                                                                                              
                                                                                                                                                                                      
REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"                                                                                                                          
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")                                                                                                                               
                                                                                                                                                                                      
if not PROJECT_DIR.exists():                                                                                                                                                          
    print("Cloning repository...")                                                                                                                                                    
    !git clone -b main $REPO_URL $PROJECT_DIR                                                                                                                                         
else:                                                                                                                                                                                 
    print("Updating repository...")                                                                                                                                                   
    %cd $PROJECT_DIR                                                                                                                                                                  
    !git pull origin main                                                                                                                                                             
                                                                                                                                                                                      
%cd $PROJECT_DIR                                                                                                                                                                      
print("Current Working Directory:", os.getcwd()) 

In [ ]:
import os
import glob
from pathlib import Path
from huggingface_hub import HfApi, login

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or not HF_TOKEN.strip():
    print("⚠️ Hãy vào menu Add-ons -> Secrets trên Kaggle và TÍCH CHỌN checkbox cạnh HF_TOKEN.")
    HF_TOKEN = input("Hoặc nhập token vào đây: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

# 1. Tạo sẵn thư mục
os.makedirs("data/indices/bm25s_index", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# 2. Kiểm tra xem đã có FAISS SQ8 và SQLite Slim trong Kaggle Input chưa
has_kaggle_dense = bool(glob.glob("/kaggle/input/**/*.faiss", recursive=True))
has_kaggle_sqlite = bool(glob.glob("/kaggle/input/**/chunks_meta*.sqlite", recursive=True))

# 3. Danh sách file cần tải từ HF Bucket (nieths/ViBioMIR)
all_files = [
    ("indices/bm25s_index/data.csc.index.npy", "data/indices/bm25s_index/data.csc.index.npy"),
    ("indices/bm25s_index/indices.csc.index.npy", "data/indices/bm25s_index/indices.csc.index.npy"),
    ("indices/bm25s_index/indptr.csc.index.npy", "data/indices/bm25s_index/indptr.csc.index.npy"),
    ("indices/bm25s_index/params.index.json", "data/indices/bm25s_index/params.index.json"),
    ("indices/bm25s_index/vocab.index.json", "data/indices/bm25s_index/vocab.index.json"),
    ("indices/bm25s_metadata.json", "data/indices/bm25s_metadata.json"),
    ("processed/pubmed_cache.jsonl", "data/processed/pubmed_cache.jsonl"),
]

if not has_kaggle_dense:
    all_files.extend([
        ("indices/dense_index_sq8.faiss", "data/indices/dense_index_sq8.faiss"),
        ("indices/dense_metadata_sq8.json", "data/indices/dense_metadata_sq8.json"),
    ])
else:
    print("✓ Đã phát hiện Dense FAISS Index trong Kaggle Input!")

if not has_kaggle_sqlite:
    all_files.append(("indices/chunks_meta_slim.sqlite", "data/indices/chunks_meta_slim.sqlite"))
else:
    print("✓ Đã phát hiện Chunks SQLite trong Kaggle Input!")

# 4. Lọc ra các file chưa có hoặc dung lượng bằng 0
missing_files = []
for remote, local in all_files:
    p = Path(local)
    if not p.exists() or p.stat().st_size == 0:
        missing_files.append((remote, local))
    else:
        print(f"✓ Đã có sẵn: {local} ({p.stat().st_size / (1024*1024):.2f} MB)")

# 5. Tải các file còn thiếu siêu tốc từ HF Bucket
if missing_files:
    print(f"\nĐang tải {len(missing_files)} file (~{sum([p.stat().st_size for _, l in missing_files if Path(l).exists()] or [0])/1024:.0f} MB còn thiếu) từ HF Bucket nieths/ViBioMIR...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=missing_files,
        token=HF_TOKEN,
    )
    print("✅ Tải xong toàn bộ các file chỉ mục cần thiết!")
else:
    print("\n✅ Tất cả các file chỉ mục (Dense SQ8 + BM25s + SQLite Slim) đã sẵn sàng!")


In [ ]:
!ls -lh data/indices

In [ ]:
# 6. Chạy inference song song trên 2 GPU T4 (mỗi shard ~600 query, có checkpoint) rồi merge
import glob
import os
import subprocess
import sys
from pathlib import Path

import pyarrow.parquet as pq
import torch
from huggingface_hub import hf_hub_download

dense_arg = "data/indices"
meta_arg = "data/indices/chunks_meta_slim.sqlite"

k_faiss = glob.glob("/kaggle/input/**/*.faiss", recursive=True)
k_sqlite = glob.glob("/kaggle/input/**/chunks_meta*.sqlite", recursive=True)
if k_faiss:
    dense_arg = str(Path(k_faiss[0]).parent)
    print(f"✓ Sử dụng Dense FAISS từ Kaggle Input: {k_faiss[0]}")
if k_sqlite:
    meta_arg = k_sqlite[0]
    print(f"✓ Sử dụng Chunks SQLite từ Kaggle Input: {k_sqlite[0]}")

# Resolve the queries file once so every shard reads identical rows in identical order.
k_query = glob.glob("/kaggle/input/**/query.parquet", recursive=True)
queries_path = Path(k_query[0]) if k_query else Path("data/raw/vibio_mir/query.parquet")
if not queries_path.exists():
    queries_path.parent.mkdir(parents=True, exist_ok=True)
    queries_path = Path(
        hf_hub_download(
            repo_id="AIGuruTinix/ViBioMIR",
            filename="query.parquet",
            repo_type="dataset",
            local_dir=str(queries_path.parent),
        )
    )

total_queries = pq.read_table(queries_path, columns=["id"]).num_rows
print(f"Tổng số query: {total_queries}")

gpu_count = torch.cuda.device_count()
if gpu_count >= 2:
    half = total_queries // 2
    shard_specs = [(0, 0, half), (1, half, total_queries - half)]
else:
    print(f"⚠️ Chỉ thấy {gpu_count} GPU — chạy 1 tiến trình duy nhất (chậm hơn ~2x). Cần accelerator GPU T4 x2.")
    shard_specs = [(0, 0, total_queries)]

procs = []
for gpu, offset, limit in shard_specs:
    shard_name = f"shard{gpu}"
    log_file = open(f"/kaggle/working/{shard_name}.log", "w")
    cmd = [
        sys.executable, "scripts/run_inference_only.py",
        "--queries", str(queries_path),
        "--dense-dir", dense_arg,
        "--chunks-meta-path", meta_arg,
        "--hybrid-top-k", "350",
        "--top-k-chunks", "150",
        "--top-k-docs", "150",
        "--max-chunks-per-doc", "3",
        "--expand-top-docs", "30",
        "--expand-max-chunks-per-doc", "8",
        "--batch-size", "32",
        "--disable-pubmed-network",
        "--offset", str(offset),
        "--limit", str(limit),
        "--checkpoint", f"outputs/submissions/{shard_name}.partial.jsonl",
        "--output-name", f"{shard_name}.json",
    ]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_HOME="/tmp/hf")
    print(f"🚀 {shard_name}: query [{offset}, {offset + limit}) trên GPU {gpu}")
    procs.append((shard_name, subprocess.Popen(cmd, env=env, stdout=log_file, stderr=subprocess.STDOUT), log_file))

failed = []
for shard_name, proc, log_file in procs:
    returncode = proc.wait()
    log_file.close()
    print(f"\n=== {shard_name} kết thúc (exit code {returncode}) ===")
    print(subprocess.run(["tail", "-n", "20", f"/kaggle/working/{shard_name}.log"], capture_output=True, text=True).stdout)
    if returncode != 0:
        failed.append(shard_name)

if failed:
    raise RuntimeError(
        f"Shard lỗi: {failed}. Partial predictions ở outputs/submissions/*.partial.jsonl — "
        "chạy lại cell này sẽ resume các query đã xong."
    )

merge_cmd = [
    sys.executable, "scripts/merge_submission_shards.py",
    "--inputs", *[f"outputs/submissions/shard{gpu}.json" for gpu, _, _ in shard_specs],
    "--output-name", "submission.json",
    "--expect", str(total_queries),
]
print("\n🔗 Merge:", " ".join(merge_cmd[1:]))
merge_result = subprocess.run(merge_cmd, text=True)
if merge_result.returncode != 0:
    raise RuntimeError("Merge thất bại — kiểm tra log của từng shard.")


In [ ]:
import base64
from IPython.display import HTML, display

zip_path = "/kaggle/working/submission.zip"
with open(zip_path, "rb") as f:
    data_b64 = base64.b64encode(f.read()).decode()

html_button = f'''
<a href="data:application/zip;base64,{data_b64}" download="submission.zip" 
   style="display:inline-block; padding:12px 24px; background:#007acc; color:white; 
          text-decoration:none; border-radius:6px; font-weight:bold; font-size:15px;">
   ⬇ BẤM VÀO ĐÂY ĐỂ TẢI SUBMISSION.ZIP VỀ MÁY
</a>
'''
display(HTML(html_button))

In [ ]:
# 8. Kiểm tra cấu trúc file submission.zip vừa tạo
import json
import zipfile
from pathlib import Path

sub_zip = Path("/kaggle/working/submission.zip")
if not sub_zip.exists():
    sub_zip = Path("submission.zip")

if sub_zip.exists():
    with zipfile.ZipFile(sub_zip, "r") as z:
        print(f"✓ File trong submission.zip: {z.namelist()}")
        with z.open("submission.json") as f:
            sample_data = json.load(f)
            print(f"✓ Tổng số queries: {len(sample_data):,}")
            if sample_data:
                q0 = sample_data[0]
                print(f"  Mẫu Query ID {q0.get('id')}:")
                print(f"    - Số docs dự đoán:   {len(q0.get('relevant_docs', []))}")
                print(f"    - Số chunks dự đoán: {len(q0.get('relevant_chunks', []))}")
                if q0.get('relevant_chunks'):
                    c0 = q0['relevant_chunks'][0]
                    print(f"    - Mẫu chunk #1: [Doc {c0.get('doc_id')}] {c0.get('chunk_text', '')[:100]}...")
else:
    print("⚠️ Chưa tìm thấy submission.zip. Hãy chạy Cell 6 trước!")
